Script de traitement d'image pour le controle qualité<br>
les fonctionnalité:<br>
validation du nom: cmd + 8 chiffres + 6 chiffres(dates)<br>
Vérification du format PNG (contenu réel)<br>
Extraction des métadonnées EXIF<br>
Redimentionnement automatique <br>
classement par date dans des sous dossiers <br>
Géneration des rapports JSON pour C#



In [1]:
!pip install piexif

In [2]:
import os
import re
import json
import shutil
import hashlib

from datetime import datetime
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import logging
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ExifTags
import piexif

In [3]:
print ("Toutes les bibliotheques sont importées avec succées")
print (f"Version OpenCV: {cv2.__version__}")
print (f"Version PIL: {Image.__version__}")

Toutes les bibliotheques sont importées avec succées
Version OpenCV: 4.13.0
Version PIL: 11.3.0


**CONFIGURATION DU PROJET**
# Nous configurons les chemins et paramètres du traitement.
#
# VARIABLES :
- DOSSIER_PHOTOS : Dossier source contenant les images
 - DOSSIER_BASE : Dossier racine pour la quarantaine
 - TAILLE_MAX : Taille maximale des images redimensionnées
 - PATTERN_NOM_IMAGE : Expression régulière pour valider les noms

In [14]:
import os
import sys

def is_colab():
  try:
    import google.colab
    return True
  except ImportError:
    return False
def is_windows():
  return sys.platform == "win32"

# if is_colab():
#  DOSSIER_PHOTOS = "/content/drive/MyDrive/ProjectsDev/Photos"
#  DOSSIER_BASE = "/content/drive/MyDrive/ProjectsDev/folders"
#  if not os.path.exists("/content/drive"):
#    from google.colab import drive
#   drive.mount("/content/drive")
if is_windows():
  DOSSIER_PHOTOS = r"C:\ProjectsDev\Photos"
  DOSSIER_BASE = r"C:\ProjectsDev\folders"
else:
  DOSSIER_PHOTOS = os.path.expanduser("~/ProjectsDev/Photos")
  DOSSIER_BASE = os.path.expanduser("~/ProjectsDev/folders")
DATE_TRAITEMENT = datetime.now().strftime("%Y%m%d")
DOSSIER_QUARANTAINE = os.path.join(DOSSIER_BASE, f"Qlte_{DATE_TRAITEMENT}")

os.makedirs(DOSSIER_PHOTOS, exist_ok=True)
os.makedirs(DOSSIER_QUARANTAINE, exist_ok=True)


In [15]:
#DOSSIER_PHOTOS = r"C:\ProjectsDev\Photos"
#DOSSIER_BASE = r"C:\ProjectsDev\folders"
#DATE_Traitement = datetime.now().strftime("%Y-%m-%d")
#DOSSIER_QUARANTAINE = os.path.join(DOSSIER_BASE, f"Qlte_{DATE_TRAITEMENT}")
TAILLE_MAX = (1024, 1024)
QUALITE_JPEG = 85
PATTERN_NOM_IMAGE = re.compile(r'^cmd(\d{8})(\d{6})\.png$', re.IGNORECASE)
LOG_FILE = "image_processing.log"
logging.basicConfig(
    level=logging.INFO,
    format= '%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print(" Configuration terminée !")
print(f" Dossier source : {DOSSIER_PHOTOS}")
print(f" Dossier quarantaine : {DOSSIER_QUARANTAINE}")
print(f" Taille max : {TAILLE_MAX[0]}x{TAILLE_MAX[1]}")
print(f" Pattern : {PATTERN_NOM_IMAGE.pattern}")

 Configuration terminée !
 Dossier source : /root/ProjectsDev/Photos
 Dossier quarantaine : /root/ProjectsDev/folders/Qlte_20260705
 Taille max : 1024x1024
 Pattern : ^cmd(\d{8})(\d{6})\.png$


**FONCTIONS DE VALIDATION**
# EXPLICATION :
 Ces fonctions valident la structure des fichiers.
#
 1. valider_date() : Vérifie que la date est au format AAMMJJ
 2. est_vrai_png() : Vérifie le magic number PNG
 3. valider_nom_fichier() : Valide le nom complet


In [16]:
from typing import Tuple

def valider_date(date_str: str) -> tuple[bool, str]:
  if len(date_str) != 6:
    return False, f"Lnogueur invalide: {date_str} doit etre en 6 Caracteres"
  try:
    jour = int(date_str[4:6])
    mois = int(date_str[2:4])
    annee = int(date_str[0:2])
    if not (0 <= annee <= 99):
      return False, f"Annee Invalide {annee}"
    if not (1<= mois <= 12):
      return False, f"Mois Invalide {mois}"
    if not (0 <= jour <= 31):
      return False, f"Jour Invalide {jour}"
    jours_par_mois = [
        31, 29 if (2000 + annee) % 4 == 0 and ((2000 + annee) % 100 != 0 or (2000 + annee) % 400 == 0) else 28, 31, 30, 31, 30, 31
    ]
    if jour > jours_par_mois[mois - 1]:
      return False, f"Jour Invalide par mois {jour}"
    date_object = datetime(2000 + annee, mois, jour)
    return True, date_object.strftime("%d-%m-%y")
  except Exception as e:
    return False, f"Erreur de Validation de date: {e}"

- Vérifie qu'un fichier est vraiment un PNG (vérification du contenu)    
    Args:
        chemin_fichier (str): Chemin complet du fichier        
    Returns:
        bool: True si le fichier est un vrai PNG
 -   Valide complètement un nom de fichier    
    Args:
        nom_fichier (str): Nom du fichier à valider        
    Returns:
        Dict: Résultat de la validation avec toutes les informations

In [17]:
def is_valid_png(chemin_fichier: str) -> bool:
  try:
    with open (chemin_fichier, 'rb') as f:
      header = f.read(8)
      return header == b'\x89PNG\r\n\x1a\n'
  except Exception:
    return False

def is_valide_nomFichier(nom_fichier: str) -> Dict[str, any]:
  resultat = {
        "valide": False,
        "nom": nom_fichier,
        "numero_reclamation": None,
        "date_brute": None,
        "date_formatee": None,
        "chemin_date": None,
        "erreur": None
    }
  match = PATTERN_NOM_IMAGE.match(nom_fichier)
  if not match:
    resultat["erreur"] = "Invalide format attendu cmd12345678300629.png "
    return resultat

  numero_reclamation = match.group(1)
  date_brute = match.group(2)

  date_valide, date_formatee = valider_date(date_brute)
  if not date_valide:
    resultat["erreur"] = f"Date Invalide {date_brute}"
    return resultat

  resultat["valide"] = True
  resultat["numero_reclamation"] = numero_reclamation
  resultat["date_brute"] = date_brute
  resultat["date_formatee"] = date_formatee

  annee = 2000 + int(date_brute[0:2])
  mois = int(date_brute[2:4])
  jour = int(date_brute[4:6])
  resultat["chemin_date"] = os.path.join(str(annee), f"{mois:02d}", f"{jour:02d}")
  return resultat


**FONCTIONS DE TRAITEMENT DES IMAGES**
# Ces fonctions traitent le contenu des images.
1. extraire_metadonnees_exif() : Extrait les métadonnées EXIF<br>
    Args:
        chemin_fichier (str): Chemin complet du fichier        
    Returns:
        Dict: Dictionnaire contenant toutes les métadonnées
2. redimensionner_image() : Redimensionne l'image si nécessaire <br>
  Redimensionne une image si elle dépasse la taille max    
    Args:
        chemin_source (str): Chemin de l'image source
        taille_max (Tuple): Taille maximale (largeur, hauteur)        
    Returns:
        Tuple[bool, Optional[str]]: (succès, chemin_redimensionne)
3. classer_par_date() : Classe l'image dans un sous-dossier par date <br>
   Déplace le fichier dans un sous-dossier selon sa date    
    Args:
        chemin_fichier (str): Chemin complet du fichier
        date_brute (str): Date au format AAMMJJ        
    Returns:
        bool: True si le classement a réussi
4. afficher_image() : Affiche l'image dans le notebook (debug)<br>
  Affiche une image dans le notebook (pour débogage)    
    Args:
        chemin (str): Chemin de l'image
        titre (str): Titre de l'affichage



In [18]:
def extraire_metadonnee_exif(chemin_fichier: str) -> Dict[str, any]:
  metadonnees = {
        "date_creation": None,
        "date_modification": None,
        "appareil": None,
        "modele": None,
        "dimensions": None,
        "taille_fichier": None,
        "hash_md5": None
    }
  try:
      stat = os.stat(chemin_fichier)
      metadonnees["taille_fichier"] = stat.st_size
      metadonnees["date_creation"] = datetime.fromtimestamp(stat.st_ctime).isoformat()
      metadonnees["date_modification"] = datetime.fromtimestamp(stat.st_mtime).isoformat()

      with open(chemin_fichier, 'rb') as f:
        metadonnees["hash_md5"] = hashlib.md5(f.read()).hexdigest()
      try:
        img = Image.open(chemin_fichier)
        metadonnees["dimensions"] = f"{img.width}x{img.height}"
        exif_data = img._getexif()
        if exif_data:
          for tag_id, value in exif_data.items():
            tag_name = ExifTags.TAGS.get(tag_id, tag_id)
            if tag_name == "Make":
              metadonnees["appareil"] = str(value)
            elif tag_name == "Model":
              metadonnees["modele"] = str(value)
            elif tag_name == "DateTimeOriginal":
              metadonnees["date_original"] = str(value)
      except Exception as e :
        logger.debug(f"Exif non disponible: {e}")

  except Exception as e:
      logger.warning(f"Erreur lors de l'extraction des métadonnées: {e}")
  return metadonnees

In [19]:
from IPython.lib.display import exists
def redimensionner_image(chemin_source: str, taille_max: Tuple[int, int] = TAILLE_MAX ) -> Tuple[bool, Optional[str]]:
  try:
    img = cv2.imread(chemin_source)
    if img is None:
      return False, None
    hauteur, largeur = img.shape[:2]
    if hauteur <= taille_max[1] and largeur <= taille_max[0]:
      return True, chemin_source
    ratio = min(taille_max[0] / largeur, taille_max[1] / hauteur)
    nouvelle_largeur = int(largeur * ratio)
    nouvelle_hauteur = int(hauteur * ratio)

    img_redim = cv2.resize(ing, (nouvelle_largeur, nouvelle_hauteur), interpolation=cv2.INTER_LANCZOS4)
    cv2.imwrite(chemin_source, img_redim, [cv2.IMWRITE_BMP_COMPRESSION, 9])
    logger.info (f"Image Redimensionee: {largeur}x{hauteur} -> {nouvelle_largeur}x{nouvelle_hauteur}")
    return True, chemin_source
  except Exception as e:
    logger.error(f"erreur redimensionnement: {e}")
    return False, None

def classer_par_date(chemin_fichier: str, date_brute: str) -> bool:
  try:
    annee = 200 + int(date_brute[0:2])
    mois = int(date_brute[2:4])
    jour = int(date_brute[4:6])
    dossier_cible = os.path.join(DOSSIER_PHOTOS, str(annee), f"{mois:02d}", f"{jour:02d}")
    Path(dossier_cible).os.mkdir(parents=True, exists_ok=True)

    nom_fichier = os.path.basename(chemin_fichier)
    chemin_cible = os.path.join(dossier_cible, nom_fichier)

    if os.path.exists(chemin_cible):
      base, ext = os.path.splittext(nom_fichier)
      compteur = 1
      while os.path.exists(chemin_cible):
        new_nom = f"{base}_{compteur}{ext}"
        chemin_cible = os.path.join(dossier_cible, new_nom)
        compteur += 1
    shutil.move(chemin_fichier, chemin_cible)
    logger.info(f"Dossier classé: {nom_fichier} -> {dossier_cible}")
    return True
  except Exception as e:
    logger.error(f"Erreur Classement: {e}")
    return False

def afficher_image(chemin: str, titre: str ="Image"):
  try:
    img = cv2.imread(chemin)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8,6))
    plt.imshow(img_rgb)
    plt.title(titre)
    plt.axis('off')
    plt.show()
  except Exception as e:
    print(f"Impossible d afficher l image: {e}")

**FONCTION PRINCIPALE DE TRAITEMENT**
# Cette fonction coordonne tout le traitement des images.
# ÉTAPES :
1. Création du dossier de quarantaine
2. Parcours de tous les fichiers du dossier
3. Validation du nom et du contenu
4. Extraction des métadonnées
5. Redimensionnement
6. Classement par date
7. Mise en quarantaine des invalides
8. Génération du rapport JSON

In [20]:

def traiter_dossier_images() -> Dict:
    resultat = {
        "success": False,
        "timestamp": datetime.now().isoformat(),
        "dossier_source": DOSSIER_PHOTOS,
        "dossier_quarantaine": DOSSIER_QUARANTAINE,
        "statistiques": {
            "total": 0,
            "valides": 0,
            "invalides": 0,
            "redimensionnees": 0,
            "classees": 0
        },
        "fichiers_valides": [],
        "fichiers_invalides": [],
        "erreurs": [],
        "rapport_json": None
    }

    try:
        Path(DOSSIER_QUARANTAINE).mkdir(parents=True, exist_ok=True)
        logger.info(f" Dossier de quarantaine : {DOSSIER_QUARANTAINE}")

        if not os.path.exists(DOSSIER_PHOTOS):
            logger.error(f" Le dossier {DOSSIER_PHOTOS} n'existe pas.")
            resultat["erreurs"].append({"erreur_globale": "Dossier source inexistant"})
            return resultat

        fichiers = os.listdir(DOSSIER_PHOTOS)
        resultat["statistiques"]["total"] = len(fichiers)
        logger.info(f" {len(fichiers)} fichiers trouvés dans le dossier")

        for fichier in fichiers:
            chemin_complet = os.path.join(DOSSIER_PHOTOS, fichier)
            if not os.path.isfile(chemin_complet):
                continue

            logger.info(f"📄 Traitement de : {fichier}")

            validation_nom = is_valide_nomFichier(fichier)

            est_png = is_valid_png(chemin_complet)

            if not validation_nom["valide"] or not est_png:
                try:
                    chemin_quarantaine = os.path.join(DOSSIER_QUARANTAINE, fichier)

                    if os.path.exists(chemin_quarantaine):
                        base, ext = os.path.splitext(fichier)
                        compteur = 1
                        while os.path.exists(chemin_quarantaine):
                            nouveau_nom = f"{base}_{compteur}{ext}"
                            chemin_quarantaine = os.path.join(DOSSIER_QUARANTAINE, nouveau_nom)
                            compteur += 1

                    shutil.move(chemin_complet, chemin_quarantaine)

                    invalide = {
                        "fichier": fichier,
                        "destination": chemin_quarantaine,
                        "raisons": []
                    }
                    if not validation_nom["valide"]:
                        invalide["raisons"].append(validation_nom["erreur"])
                    if not est_png:
                        invalide["raisons"].append("Fichier PNG invalide (contenu non conforme)")

                    resultat["fichiers_invalides"].append(invalide)
                    resultat["statistiques"]["invalides"] += 1
                    logger.warning(f" Fichier invalide : {fichier} → Quarantaine")

                except Exception as e:
                    resultat["erreurs"].append({"fichier": fichier, "erreur": str(e)})
                    logger.error(f" Erreur déplacement : {fichier} - {e}")

                continue

            metadonnees = extraire_metadonnee_exif(chemin_complet)
            redim_ok, _ = redimensionner_image(chemin_complet)
            if redim_ok:
                resultat["statistiques"]["redimensionnees"] += 1
            else:
                resultat["erreurs"].append({
                    "fichier": fichier,
                    "erreur": "Échec du redimensionnement"
                })

            if validation_nom["chemin_date"]:
                classement_ok = classer_par_date(chemin_complet, validation_nom["date_brute"])
                if classement_ok:
                    resultat["statistiques"]["classees"] += 1
                else:
                    resultat["erreurs"].append({
                        "fichier": fichier,
                        "erreur": "Échec du classement"
                    })

            valide = {
                "fichier": fichier,
                "numero_reclamation": validation_nom["numero_reclamation"],
                "date_reception": validation_nom["date_formatee"],
                "metadonnees": metadonnees
            }
            resultat["fichiers_valides"].append(valide)
            resultat["statistiques"]["valides"] += 1
            logger.info(f" Fichier valide : {fichier}")

        # 4. rapport JSON
        rapport_json = os.path.join(DOSSIER_PHOTOS, "rapport_traitement.json")
        with open(rapport_json, 'w', encoding='utf-8') as f:
            json.dump(resultat, f, ensure_ascii=False, indent=2)

        resultat["rapport_json"] = rapport_json
        resultat["success"] = True

        # 5. Affichage du résumé
        logger.info("=" * 60)
        logger.info(" TRAITEMENT TERMINÉ AVEC SUCCÈS")
        logger.info(f" Statistiques :")
        logger.info(f"  - Total fichiers : {resultat['statistiques']['total']}")
        logger.info(f"  - Valides : {resultat['statistiques']['valides']}")
        logger.info(f"  - Invalides : {resultat['statistiques']['invalides']}")
        logger.info(f"  - Redimensionnées : {resultat['statistiques']['redimensionnees']}")
        logger.info(f"  - Classées : {resultat['statistiques']['classees']}")
        logger.info(f" Rapport JSON : {rapport_json}")
        logger.info("=" * 60)

        return resultat

    except Exception as e:
        logger.error(f"Erreur générale : {e}")
        resultat["erreurs"].append({"erreur_globale": str(e)})
        return resultat

print(" Fonction principale prête !")

 Fonction principale prête !


**EXÉCUTION DU TRAITEMENT**
# Cette cellule lance le traitement complet des images.

ATTENTION : Assurez-vous que le dossier DOSSIER_PHOTOS existe
et contient des images avant d'exécuter cette cellule.

In [21]:
resultat = traiter_dossier_images()

if resultat["success"]:
    print("\n" + "--" * 60)
    print("RÉSULTATS DÉTAILLÉS")
    print("--" * 60)
    print(f" Succès : {resultat['success']}")
    print(f" Dossier source : {resultat['dossier_source']}")
    print(f" Dossier quarantaine : {resultat['dossier_quarantaine']}")
    print(f" Rapport JSON : {resultat['rapport_json']}")
    print("\n Statistiques :")
    for key, value in resultat['statistiques'].items():
        print(f"  - {key} : {value}")

    if resultat['fichiers_valides']:
        print(f"\n {len(resultat['fichiers_valides'])} fichiers valides :")
        for f in resultat['fichiers_valides'][:5]:  # on affiche les cinq premiers
            print(f"  - {f['fichier']} (Réclamation: {f['numero_reclamation']}, Date: {f['date_reception']})")
        if len(resultat['fichiers_valides']) > 5:
            print(f"  ... et {len(resultat['fichiers_valides']) - 5} autres")

    if resultat['fichiers_invalides']:
        print(f"\n {len(resultat['fichiers_invalides'])} fichiers invalides :")
        for f in resultat['fichiers_invalides'][:5]:
            print(f"  - {f['fichier']} → {os.path.basename(f['destination'])}")
            for raison in f['raisons']:
                print(f"       {raison}")
        if len(resultat['fichiers_invalides']) > 5:
            print(f"  ... et {len(resultat['fichiers_invalides']) - 5} autres")

    if resultat['erreurs']:
        print(f"\n {len(resultat['erreurs'])} erreurs :")
        for err in resultat['erreurs'][:5]:
            print(f"  - {err}")

    print("\n" + "--" * 60)
    print("Traitement terminé avec succès !")
else:
    print(" Le traitement a échoué.")
    print(f"Erreurs : {resultat.get('erreurs', [])}")

if __name__ == "__main__":
    resultat = traiter_dossier_images()



------------------------------------------------------------------------------------------------------------------------
RÉSULTATS DÉTAILLÉS
------------------------------------------------------------------------------------------------------------------------
 Succès : True
 Dossier source : /root/ProjectsDev/Photos
 Dossier quarantaine : /root/ProjectsDev/folders/Qlte_20260705
 Rapport JSON : /root/ProjectsDev/Photos/rapport_traitement.json

 Statistiques :
  - total : 0
  - valides : 0
  - invalides : 0
  - redimensionnees : 0
  - classees : 0

------------------------------------------------------------------------------------------------------------------------
Traitement terminé avec succès !


In [22]:
if resultat["success"] and resultat["fichiers_valides"]:
  print("Affichage :")
  for i, f in enumerate(resultat["fichiers_valides"][:3]):
    chemin_img = os.path.join(DOSSIER_PHOTOS, f["fichier"])
    if os.path.exists(chemin_img):
      print(f"\nImage {i +1}: {f['fichier']}")
      try:
        img = cv2.imread(chemin_img)
        if img is not None:
           img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
           plt.figure(figsize=(6, 4))
           plt.imshow(img_rgb)
           plt.title(f"{f['fichier']}\nRéclamation: {f['numero_reclamation']}")
           plt.axis('off')
           plt.show()
        else:
           print(" Impossible de lire l'image")
      except Exception as e:
           print(f"Erreur d'affichage : {e}")
    else:
       print(f"Image introuvable : {chemin_img}")


In [23]:
from google.colab import files

if resultat["success"] and resultat["rapport_json"]:
    print("Téléchargement du rapport JSON...")
    files.download(resultat["rapport_json"])
    print("Rapport JSON téléchargé !")

if os.path.exists(LOG_FILE):
    print("Téléchargement des logs...")
    files.download(LOG_FILE)
    print("Logs téléchargés !")

Téléchargement du rapport JSON...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Rapport JSON téléchargé !
Téléchargement des logs...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Logs téléchargés !
